# Import packages

In [1]:
import os
import requests
import re
import pickle
import zipfile

import pandas as pd
from io import StringIO

# Custom functions

In [2]:
from tools.create_jaspar_filters import obtain_taxonomy_id, obtain_all_cell_lines, \
    cell_lines_df_clear_save, extract_cell_line_names, cell_line_names_to_model_id,expression_df_to_expression_tf_df_cell_lines

# Work directory setup

In [3]:
save_direct_filters = 'input/filters'
save_direct_cell_lines = 'input/cell_lines'
save_direct_jaspar_dataset = 'input/jaspar_dataset'

for dir in [save_direct_filters, save_direct_cell_lines, save_direct_jaspar_dataset]:
    os.makedirs(dir, exist_ok=True)

# Download datasets


## Human protein atlas
This is for all the normal cell lines
* RNA expression levels per gene and cell type: https://www.proteinatlas.org/download/tsv/rna_single_cell_type.tsv.zip

In [ ]:
url_protein_atlas = 'https://www.proteinatlas.org/download/tsv/rna_single_cell_type.tsv.zip'
save_path_protein_atlas_zip = os.path.join(save_direct_filters, 'rna_single_cell.zip')
save_path_protein_atlas = os.path.join(save_direct_filters, 'rna_single_cell_type.tsv')

if not os.path.exists(save_path_protein_atlas):
    r = requests.get(url_protein_atlas)

    with open(save_path_protein_atlas_zip, 'wb') as f:
        f.write(r.content)

    with zipfile.ZipFile(save_path_protein_atlas_zip, 'r') as zip_ref: #Unzip
        zip_ref.extractall(save_direct_filters)

    os.remove(save_path_protein_atlas_zip) #Delete zip

    print(f'Human protein atlas file is downloaded at {save_path_protein_atlas}')
else:
    print(f'Human protein atlas file is already available at {save_path_protein_atlas}')

## Cellosaurus
This provides all the cell lines of interests
* For the downstream, only the cancer cell lines need this and the cell lines will meet a dead end, so you don't need to download normal cell lines here

Cellosaurus API methods & exploration: https://api.cellosaurus.org/api-methods#/

### Obtain taxonomy code
This can be used later to filter cell lines by species

In [ ]:
species_search_term = 'Homo sapiens'
tax_id = obtain_taxonomy_id(species_search_term)

### Obtain cell lines

In [ ]:
search_list = [
    {
        'search_term': 'melanoma',
        'search_term_type': 'di'  #Because melanoma is a disease
    }
    #The search below is commented out because normal cell lines later go from another route. I keep it here in case for future reference
    # {
    #     'search_term': 'astrocyte',
    #     'search_term_type': 'cell-type'  #Because astrocyte is a cell type
    # }
]

for search_dict in search_list:
    search_term = search_dict['search_term']
    save_dir_cell_line_csv = os.path.join(save_direct_cell_lines, f'cell_lines_{search_term}.csv')

    if not os.path.exists(save_dir_cell_line_csv):
        print(f'>>>> Working on {search_term}')
        search_term_type = search_dict['search_term_type']

        all_cell_lines_data = obtain_all_cell_lines(
            search_term,
            search_term_type,
            tax_id,
            1000
        )

        cell_line_df = cell_lines_df_clear_save(all_cell_lines_data, search_term, save_direct_cell_lines)

        print(f'>>>> All {search_term} cell lines dataset has been saved at {save_dir_cell_line_csv}')
    else:
        print(f'>>>> All {search_term} cell lines dataset is already available at {save_dir_cell_line_csv}')

## DepMap
This provides all the cancer cell lines and expression data

DepMap API exploration: https://depmap.org/portal/api/

In [4]:
url_depmap = "https://depmap.org/portal/api/download/files"

r = requests.get(url_depmap)
files_df = pd.read_csv(StringIO(r.text))  #This gives you all the downloadable files

### Model.csv

In [5]:
filtered_df = (
    files_df[files_df['filename'].str.contains('Model.csv', case=False, na=False)]
    .sort_values('release_date', ascending=False)
    .reset_index(drop=True)
)

filtered_df

,release,release_date,filename,url,md5_hash
0,DepMap Public 25Q3,2025-09-25,Model.csv,https://storage.googleapis.com/depmap-external...,af4472ab734ea3aec974d992b504c7e5
1,DepMap Public 25Q2,2025-06-27,Model.csv,https://storage.googleapis.com/depmap-external...,9f7b51859a22c6fbe4c8cce43ae4bdf5
2,DepMap Public 24Q4,2024-12-16,Model.csv,https://ndownloader.figshare.com/files/51065297,675210d17675f3517b0ce39a3c274f16
3,DepMap Public 24Q2,2024-05-22,Model.csv,https://ndownloader.figshare.com/files/46489732,e5b60d1ce7636542d93f45494019eded
4,DepMap Public 23Q4,2023-11-20,Model.csv,https://ndownloader.figshare.com/files/43346895,c14f90f02609b9507c8b31e6d5db309b
5,DepMap Public 23Q2,2023-05-08,Model.csv,https://ndownloader.figshare.com/files/40448834,05edfe0f5e687d5b55e5affb17e8545f


In [20]:
model_url = filtered_df.iloc[0]['url']
save_dir_model_csv = os.path.join(save_direct_filters, 'Model.csv')

if not os.path.exists(save_dir_model_csv):
    r = requests.get(model_url, stream=True)

    with open(save_dir_model_csv, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f'Model.csv is downloaded at {save_dir_model_csv}')
else:
    print(f'Model.csv is already avialable at {save_dir_model_csv}')

model_df = pd.read_csv(save_dir_model_csv)
print('Model.csv has {} cell lines'.format(len(set(model_df['StrippedCellLineName']))))

Model.csv is already avialable at input/filters/Model.csv
Model.csv has 2132 cell lines


### Expression dataset

In [7]:
filtered_df = (
    files_df[files_df['filename'].str.contains('OmicsExpressionProteinCodingGenesTPMLogp1.csv', case=False, na=False)]
    .sort_values('release_date', ascending=False)
    .reset_index(drop=True)
)

filtered_df

,release,release_date,filename,url,md5_hash
0,DepMap Public 25Q2,2025-06-27,OmicsExpressionProteinCodingGenesTPMLogp1.csv,https://storage.googleapis.com/depmap-external...,f47da0b6f8f3bbd4fe62e272e53bba4f
1,DepMap Public 24Q4,2024-12-16,OmicsExpressionProteinCodingGenesTPMLogp1.csv,https://ndownloader.figshare.com/files/51065489,71794802b750ce77c422dad0720a40af
2,DepMap Public 24Q2,2024-05-22,OmicsExpressionProteinCodingGenesTPMLogp1.csv,https://ndownloader.figshare.com/files/46490878,f00c434db50d811544ac4d047003979f
3,DepMap Public 23Q4,2023-11-20,OmicsExpressionProteinCodingGenesTPMLogp1.csv,https://ndownloader.figshare.com/files/43347204,9402aa25a19279bb20a5d6cf8791a88f
4,DepMap Public 23Q2,2023-05-08,OmicsExpressionProteinCodingGenesTPMLogp1.csv,https://ndownloader.figshare.com/files/40449128,e785fe0e00f0f4fae1e7d841336970a0
5,DepMap Public 22Q4,2022-12-01,OmicsExpressionProteinCodingGenesTPMLogp1.csv,https://ndownloader.figshare.com/files/38357462,0f502b8300df0c33bc5ac260110b8ddd


In [11]:
url_expression = filtered_df.iloc[0]['url']
save_dir_expression_csv = os.path.join(save_direct_filters, 'OmicsExpressionProteinCodingGenesTPMLogp1.csv')

if not os.path.exists(save_dir_expression_csv):
    r = requests.get(url_expression, stream=True)

    with open(save_dir_expression_csv, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f'OmicsExpressionProteinCodingGenesTPMLogp1.csv is downloaded at {save_dir_expression_csv}')
else:
    print(f'OmicsExpressionProteinCodingGenesTPMLogp1.csv is already available at {save_dir_expression_csv}')

expression_df = pd.read_csv(save_dir_expression_csv)  # This will take a while since the file is pretty big

rows, cols = expression_df.shape
print(f'expression_df has {rows} cell lines (ModelID) and {cols - 1} expression data of genes')

del expression_df  #This is to save on the memory since it's too large. You can reassign it later when you need it

OmicsExpressionProteinCodingGenesTPMLogp1.csv is already available at input/filters/OmicsExpressionProteinCodingGenesTPMLogp1.csv
expression_df has 1684 cell lines (ModelID) and 19205 expression data of genes


## JASPAR
This provides all the motifs needed for fimo's motif scanning

JASPAR: https://jaspar.elixir.no/downloads/
* Go for
    * Latest release
    * `CORE PFMs` collection for experimentally validated TF binding motifs with high quality
    * `Vertebrates` as the taxonomic group
    * `Single batch file (txt)`
    * `non-redundant` removes duplicate motifs for the same TF
    * `MEME` format can be used natively with fimo

In [13]:
url_jaspar = 'https://jaspar.elixir.no/download/data/2026/CORE/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme.txt'
file_path_jaspar = os.path.join(save_direct_jaspar_dataset, url_jaspar.split('/')[-1].split('.')[0] + '_original.txt')

if not os.path.exists(file_path_jaspar):
    r = requests.get(url_jaspar)

    with open(file_path_jaspar, "wb") as f:
        f.write(r.content)

    print(f'JASPAR dataset is downloaded at {file_path_jaspar}')
else:
    print(f'JASPAR dataset is already available at {file_path_jaspar}')

JASPAR dataset is already available at input/jaspar_dataset/JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme_original.txt


# Generate filters

## Filter cell_lines.csv → cell_line_names

### Melanoma

In [14]:
cell_line_df_melanoma = pd.read_csv('input/cell_lines/cell_lines_melanoma.csv')

cell_line_names_melanoma = extract_cell_line_names(cell_line_df_melanoma)
cell_line_names_melanoma

[' clone-1',
 ' clone-2',
 ' strain 19-4',
 ' subline S1',
 '00.08',
 '0008',
 '01.12',
 '0112',
 '02.02',
 '0202',
 '0342 mel',
 '0342mel',
 '04.01',
 '04.07',
 '0401',
 '0407',
 '05.18',
 '0518',
 '06.04',
 '06.07',
 '0604',
 '0607',
 '07.16',
 '0716',
 '0PP20',
 '10 CM',
 '10049',
 '1007P',
 '10092',
 '10101',
 '1011',
 '1011 mel',
 '1011-MEL',
 '1011-mel',
 '1011MEL',
 '1011mel',
 '10164',
 '10170',
 '10214',
 '10222',
 '10249M',
 '10266',
 '10378',
 '10417',
 '10538',
 '10538P',
 '1061 mel',
 '1061mel',
 '1067 mel',
 '1067mel',
 '1074 mel',
 '1074-MEL',
 '1074-mel',
 '1074MEL',
 '1074mel',
 '1088',
 '1088 Mel',
 '1088 mel',
 '1088-MEL',
 '1088-mel',
 '1088MEL',
 '1088Mel',
 '1088mel',
 '10CM',
 '11 CM',
 '1102',
 '1102 mel',
 '1102-MEL',
 '1102-mel',
 '1102MEL',
 '1102mel',
 '1106 mel',
 '1106-mel',
 '1106MEL',
 '1106mel',
 '1123',
 '1123-MEL',
 '1123-mel',
 '1123MEL',
 '1123mel',
 '1141 mel',
 '1141mel',
 '1143',
 '1143 mel',
 '1143-MEL',
 '1143-mel',
 '1143MEL',
 '1143mel',
 '11

### Melanoma brain metastasis (MBM)

In [15]:
df_mbm = cell_line_df_melanoma[
    cell_line_df_melanoma['organ'].str.contains('brain', case=False, na=False) &
    cell_line_df_melanoma['site_type'].str.contains('metastatic', case=False, na=False)
    ]  # These are all the melanoma that metastasizes to the brain

df_mbm = df_mbm.reset_index(drop=True)
df_mbm.to_csv('input/cell_lines/cell_lines_mbm.csv', index=False)

df_mbm.head()

,cell_line_id,cell_line_name,organ,site_type,disease,age,sex,category,species
0,CVCL_E6EY,"CCLF-KL1105T,CCLF_KL1105T,CPDM_0603X","Brain, left temporal cortex",Metastatic,Melanoma,74Y,Male,Cancer cell line,Homo sapiens (Human)
1,CVCL_E6F6,"CCLF-KL1187T,CCLF_KL1187T,CPDM_0762X","Brain, left temporal cortex",Metastatic,Melanoma,73Y,Female,Cancer cell line,Homo sapiens (Human)
2,CVCL_D3Z0,"CCLF-MELM_0026-T-B,CCLF_MELM_0026_T_B",Brain,Metastatic,Melanoma,58Y,Female,Cancer cell line,Homo sapiens (Human)
3,CVCL_X382,"COLO 297F,COLO #297,Colorado 297",Brain,Metastatic,Melanoma,76Y,Male,Cancer cell line,Homo sapiens (Human)
4,CVCL_1134,"COLO 792,COLO-792,COLO #792,COLO792,Colorado 792",Brain,Metastatic,Melanoma,62Y,Male,Cancer cell line,Homo sapiens (Human)


In [16]:
cell_line_names_mbm = extract_cell_line_names(df_mbm)
cell_line_names_mbm

['06.07',
 '0607',
 '875',
 'A-875',
 'A2018',
 'A875',
 'C8',
 'CCLF-KL1105T',
 'CCLF-KL1187T',
 'CCLF-MELM_0026-T-B',
 'CCLFKL1105T',
 'CCLFKL1187T',
 'CCLFMELM0026TB',
 'CCLF_KL1105T',
 'CCLF_KL1187T',
 'CCLF_MELM_0026_T_B',
 'COLO #297',
 'COLO #792',
 'COLO 297F',
 'COLO 792',
 'COLO-792',
 'COLO297',
 'COLO297F',
 'COLO792',
 'CPDM0603X',
 'CPDM0762X',
 'CPDM_0603X',
 'CPDM_0762X',
 'Colorado 297',
 'Colorado 792',
 'Colorado297',
 'Colorado792',
 'D5',
 'Ed141.MEL',
 'Ed141MEL',
 'HTZ 19',
 'HTZ 19-dM',
 'HTZ-19',
 'HTZ-19 dM',
 'HTZ19',
 'HTZ19dM',
 'LB3080-MELA',
 'LB3080MELA',
 'LM-MEL-24',
 'LM-MEL-45',
 'LM-MEL-7',
 'LM-MEL-71',
 'LM-MEL-77',
 'LM-MEL-8',
 'LM-Mel-24',
 'LM-Mel-45',
 'LM-Mel-7',
 'LM-Mel-71',
 'LM-Mel-77',
 'LM-Mel-8',
 'LMMEL24',
 'LMMEL45',
 'LMMEL7',
 'LMMEL71',
 'LMMEL77',
 'LMMEL8',
 'LMMel24',
 'LMMel45',
 'LMMel7',
 'LMMel71',
 'LMMel77',
 'LMMel8',
 'Ludwig Melbourne-MELanoma-24',
 'Ludwig Melbourne-MELanoma-45',
 'Ludwig Melbourne-MELanoma-7',
 'Lu

### Uveal melanoma (UM)

In [17]:
df_uveal = cell_line_df_melanoma[
    cell_line_df_melanoma['disease'].str.contains('uvea', case=False, na=False)
]  #These are all the uveal melanoma, including the in situ and metastatic ones that can go to various locations (can be further specified)

df_uveal = df_uveal.reset_index(drop=True)
df_uveal.to_csv('input/cell_lines/cell_lines_um.csv', index=False)

df_uveal.head()

,cell_line_id,cell_line_name,organ,site_type,disease,age,sex,category,species
0,CVCL_VU92,"BB90-MEL,BB 90 MEL,DEZE",NaN,NaN,"Uveal melanoma,Uveal melanoma",Age unspecified,Male,Cancer cell line,Homo sapiens (Human)
1,CVCL_8471,C918,NaN,NaN,"Uveal melanoma,Uveal melanoma",60Y,Female,Cancer cell line,Homo sapiens (Human)
2,CVCL_C7UB,1141 mel,"Eye, uvea",In situ,"Uveal melanoma,Uveal melanoma",Age unspecified,Sex unspecified,Cancer cell line,Homo sapiens (Human)
3,CVCL_6900,"EOM-29,EOM29","Eye, uvea",In situ,"Uveal melanoma,Uveal melanoma",87Y,Male,Cancer cell line,Homo sapiens (Human)
4,CVCL_6901,"EOM-3,EOM3","Eye, uvea",In situ,"Uveal melanoma,Uveal melanoma",62Y,Male,Cancer cell line,Homo sapiens (Human)


In [18]:
cell_line_names_um = extract_cell_line_names(df_uveal)
cell_line_names_um

['1141 mel',
 '1141mel',
 '15765 mel',
 '15765mel',
 '157d',
 '196b',
 '2130 mel',
 '2130mel',
 '37165 mel',
 '37165mel',
 '4022 mel',
 '4022mel',
 '4330 mel',
 '4330mel',
 '48409 mel',
 '48409mel',
 '92-1 [Human uveal melanoma]',
 '92-2 [Human uveal melanoma]',
 '92.1',
 '92.1-A',
 '92.1-B',
 '92.2',
 '921',
 '921A',
 '921B',
 '921Humanuvealmelanoma',
 '922',
 '922Humanuvealmelanoma',
 '92_1',
 '92_2',
 'BB 90 MEL',
 'BB90-MEL',
 'BB90MEL',
 'C918',
 'DEZE',
 'EOM-29',
 'EOM-3',
 'EOM29',
 'EOM3',
 'EST128',
 'HL165',
 'IPC 211',
 'IPC 227F',
 'IPC 227e',
 'IPC 281',
 'IPC-211',
 'IPC-227E',
 'IPC-227F',
 'IPC-281',
 'IPC211',
 'IPC227E',
 'IPC227F',
 'IPC227e',
 'IPC281',
 'M619',
 'MEL 290',
 'MEL15765',
 'MEL20-06-039',
 'MEL20-06-045',
 'MEL20-07-070',
 'MEL2006039',
 'MEL2006045',
 'MEL2007070',
 'MEL202',
 'MEL270',
 'MEL285',
 'MEL290',
 'MKT-BR',
 'MKTBR',
 'MM28',
 'MM33',
 'MM66',
 'MP38',
 'MP41',
 'MP41 / GFP',
 'MP41 / Luciferase',
 'MP41 / Luciferase & GFP',
 'MP41 / RFP

## cell_line_names → model_id_cell_line

In [21]:
model_id_melanoma = cell_line_names_to_model_id(cell_line_names_melanoma, model_df)
model_id_mbm = cell_line_names_to_model_id(cell_line_names_mbm, model_df)
model_id_um = cell_line_names_to_model_id(cell_line_names_um, model_df)

## expression_df → expression_tf_df_cell_lines
For each cell line
1. expression_df —[model_id_cell_line]→ expression_df_cell_lines
2. expression_df_cell_lines —[jaspar_tf]→ expression_tf_df_cell_lines

These 2 steps are done together to avoid consuming lots of memories

The recommended workflow is to use vanilla JASPAR to do fimo motif scanning and then use expression_tf_df_cell_lines to filter out the results

#### Obtain all TF in Jaspar dataset

In [22]:
tf_names = []

with open(file_path_jaspar) as motif_database:
    for line in motif_database:
        if line.startswith('MOTIF'):
            parts = line.strip().split()
            if len(parts) >= 3:
                tf_names.append(parts[2])  # third column is TF name

print(f'Original JASPAR dataset has {len(tf_names)} TF')

Original JASPAR dataset has 1019 TF


In [23]:
expression_df = pd.read_csv(
    'input/filters/OmicsExpressionProteinCodingGenesTPMLogp1.csv')  #This will take a while since the file is pretty big

In [24]:
# Use the common cutoff: log2(TPM+1) ≥ 1 (i.e., TPM ≥ 1) in at least 50% of cell lines. If so, it means the TF is expressed in melanoma
TPM_LOGP1_threshold = 1.0
FRACTION_REQUIRED = 0.5

In [25]:
variable_dict = {
    'melanoma': model_id_melanoma,
    'mbm': model_id_mbm,
    'um': model_id_um
}

for key, value in variable_dict.items():
    expression_df_to_expression_tf_df_cell_lines(expression_df, value, tf_names, TPM_LOGP1_threshold, FRACTION_REQUIRED, key)

Found 449 TF from JASPAR dataset for TF expressed in specific MELANOMA cell lines
Files saved for names of TF expressed in MELANOMA cell lines
Found 447 TF from JASPAR dataset for TF expressed in specific MBM cell lines
Files saved for names of TF expressed in MBM cell lines
Found 427 TF from JASPAR dataset for TF expressed in specific UM cell lines
Files saved for names of TF expressed in UM cell lines


### Astrocyte (AC)
It only goes over expression_df → expression_df_cell_lines since the human protein atlas expression matrix is already organized

In [26]:
df_protein_atlas = pd.read_csv('input/filters/rna_single_cell_type.tsv', sep='\t')

In [27]:
ncpm_threshold = 1 #This is a loose filter for a gene to be expressed

df_astrocyte = df_protein_atlas[
    (df_protein_atlas["Cell type"] == "astrocytes") &
    (df_protein_atlas["nCPM"] >= ncpm_threshold) &
    (df_protein_atlas["Gene name"].isin(tf_names))
]

In [28]:
gene_list_astrocyte = df_astrocyte["Gene name"].unique().tolist()

txt_path_astrocyte = os.path.join('input/filters',"tf_ac.txt")

with open(txt_path_astrocyte, "w") as f:
    for gene in gene_list_astrocyte:
        f.write(gene + "\n")

pkl_path_astrocyte =  os.path.join('input/filters',"tf_ac.pkl")

with open(pkl_path_astrocyte, "wb") as f:
    pickle.dump(gene_list_astrocyte, f)

## jaspar —[expressed_tf_cell_line]→ jaspar_cell_line
jaspar_cell_line is not recommended for fimo scanning since it might be too narrow for such an early step, so using the TF name list to filter on the JASPAR dataset is neglected here